# Predicted restaurant busyness across Manhattan

Builds the three-panel figure for Section IV-G of the Tablé paper: predicted
busyness for 300 restaurants spread across Manhattan, at **08:00 / 14:00 /
20:00** on a Friday in July.

The notebook calls the **deployed** model — `busyness_xgboost_pipeline.joblib`
through `BusynessModelService.predict`, the same entry point the FastAPI
service's `/predict/busyness` uses — so the figure reports what the running
system actually returns, not a reimplementation.

**Requirements:** `scikit-learn==1.8.0` (the version the pipeline was fitted
with), `xgboost`, `pandas`, `pyarrow`, `joblib`, `pyshp`, `pyproj`,
`matplotlib`. Everything else ships in the repo — no network access and no
basemap service is needed; the island outline comes from the TLC taxi-zone
shapefile already in `ml-pipeline/fastapi-app/data/`.

**Why 300 spread venues rather than the seeded database?** `generate-seed.js`
selects the 300 restaurants *nearest* Times Square, so the seeded set occupies a
~350 m disc and cannot produce a legible island-wide map. These 300 are drawn
from the modelling universe by proportional allocation across taxi zones, with a
minimum separation enforced between markers. Every one carries real venue
attributes; none relies on median imputation.

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapefile                      # pyshp
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon as MplPolygon
from pyproj import Transformer

%matplotlib inline

# --- point this at the repository -------------------------------------------
# Auto-detects when the notebook sits anywhere inside the repo; otherwise set
# REPO_ROOT by hand.
REPO_ROOT = None
if REPO_ROOT is None:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "ml-pipeline" / "fastapi-app" / "model_service.py").exists():
            REPO_ROOT = candidate
            break
if REPO_ROOT is None:
    raise SystemExit(
        "Set REPO_ROOT to your comp47360-team2 checkout, e.g.\n"
        "    REPO_ROOT = Path(r'C:/Users/you/Documents/GitHub/comp47360-team2')"
    )

FASTAPI_APP = Path(REPO_ROOT) / "ml-pipeline" / "fastapi-app"
OUT_DIR = Path.cwd()
sys.path.insert(0, str(FASTAPI_APP))
print("repo      :", REPO_ROOT)
print("model dir :", FASTAPI_APP)

In [ ]:
# --- prediction setting ------------------------------------------------------
HOURS = [8, 14, 20]
WEEKDAY = 4          # Monday=0 ... Friday=4, matching the service's convention
MONTH = 7            # July, matching the shipped TLC aggregates

# --- marker selection --------------------------------------------------------
TARGET = 300         # number of restaurants to plot
MIN_SEP = 130.0      # metres; no two markers may sit closer than this

# Manhattan bounding box, used only to reject corrupted geocodes.
# Four of the 2,815 venues in restaurant_features.parquet are mis-geocoded
# (three land in Brooklyn, one in Palo Alto).
LAT_MIN, LAT_MAX = 40.68, 40.88
LON_MIN, LON_MAX = -74.03, -73.90

CLASS_COLOURS = {0: "#1f4e79", 1: "#e8a33d", 2: "#b5232b"}
CLASS_NAMES = {0: "No Wait", 1: "Queue Required", 2: "Severe Queue"}

## 1. Load the deployed model

In [ ]:
from model_service import BusynessModelService, RestaurantFeatures

svc = BusynessModelService()
feat = svc.restaurant_features

uni = feat[
    feat.latitude.between(LAT_MIN, LAT_MAX)
    & feat.longitude.between(LON_MIN, LON_MAX)
].copy()
uni["taxi_zone_id"] = uni["taxi_zone_id"].astype(int)

print(f"modelling universe : {len(feat)} venues")
print(f"usable geocodes    : {len(uni)}  ({len(feat) - len(uni)} dropped)")
print(f"taxi zones         : {uni.taxi_zone_id.nunique()}")

## 2. Choose 300 restaurants spread across the island

Two constraints pull against each other: the sample should still *look* like
Manhattan's real restaurant distribution, and no two markers should overlap.

- Each taxi zone gets a quota proportional to how many restaurants it holds
  (floor of 1, so no zone disappears).
- Within a zone, each pick is the candidate farthest from everything already
  chosen **anywhere on the island**, and is rejected outright if it lands within
  `MIN_SEP` of an existing marker.
- Quota a dense zone cannot fill at that spacing is handed to zones with room.

In [ ]:
def to_metres(lat, lon):
    """Local equirectangular projection — accurate enough over one island."""
    lat0 = np.deg2rad(40.78)
    return (
        np.deg2rad(lon) * np.cos(lat0) * 6371000.0,
        np.deg2rad(lat) * 6371000.0,
    )


uni["mx"], uni["my"] = to_metres(uni.latitude.values, uni.longitude.values)

# proportional quota per zone
counts = uni.groupby("taxi_zone_id").size()
raw = counts / counts.sum() * TARGET
quota = np.floor(raw).astype(int).clip(lower=1)

while quota.sum() < TARGET:
    for z in (raw - quota).sort_values(ascending=False).index:
        if quota[z] < counts[z]:
            quota[z] += 1
            break
    else:
        break
while quota.sum() > TARGET:
    for z in (raw - quota).sort_values().index:
        if quota[z] > 1:
            quota[z] -= 1
            break


def sweep(candidates, selected_pts, want):
    """Greedy farthest-point picks honouring the global MIN_SEP floor."""
    P = candidates[["mx", "my"]].values
    d = (
        np.sqrt(((P[:, None] - selected_pts[None]) ** 2).sum(2)).min(1)
        if len(selected_pts)
        else np.full(len(candidates), np.inf)
    )
    out = []
    for _ in range(want):
        j = int(np.argmax(d))
        if d[j] < MIN_SEP:
            break
        out.append(candidates.index[j])
        selected_pts = np.vstack([selected_pts, P[j]])
        d = np.minimum(d, np.sqrt(((P - P[j]) ** 2).sum(1)))
    return out, selected_pts


chosen, pts = [], np.empty((0, 2))
for z in counts.sort_values(ascending=False).index:
    got, pts = sweep(uni[uni.taxi_zone_id == z], pts, int(quota[z]))
    chosen += got

# redistribute whatever the dense zones could not fit
for z in counts.sort_values(ascending=False).index:
    if len(chosen) >= TARGET:
        break
    spare = uni[(uni.taxi_zone_id == z) & (~uni.index.isin(chosen))]
    if spare.empty:
        continue
    got, pts = sweep(spare, pts, TARGET - len(chosen))
    chosen += got

picked = uni.loc[chosen].copy()

D = np.sqrt(((pts[:, None] - pts[None]) ** 2).sum(2))
np.fill_diagonal(D, np.inf)
nn = D.min(1)

print(f"selected           : {len(picked)} venues")
print(f"taxi zones covered : {picked.taxi_zone_id.nunique()}")
print(f"nearest neighbour  : min {nn.min():.0f} m   "
      f"median {np.median(nn):.0f} m   max {nn.max():.0f} m")
print(f"bounding box       : lat {picked.latitude.min():.4f}..{picked.latitude.max():.4f}   "
      f"lon {picked.longitude.min():.4f}..{picked.longitude.max():.4f}")

## 3. Predict

One call per venue per hour, straight through the service class. Roughly 900
predictions; inference itself is ~12 µs a row, so this is dominated by Python
overhead rather than the model.

In [ ]:
rows = []
for r in picked.itertuples():
    for hour in HOURS:
        p = svc.predict(
            hour=hour,
            weekday=WEEKDAY,
            month=MONTH,
            restaurant_id=str(r.restaurant_id),
            features=RestaurantFeatures(
                latitude=float(r.latitude),
                longitude=float(r.longitude),
                capacity=None if pd.isna(r.capacity) else int(r.capacity),
            ),
        )
        rows.append({
            "restaurant_id": r.restaurant_id,
            "name": r.original_name,
            "latitude": float(r.latitude),
            "longitude": float(r.longitude),
            "taxi_zone_id": int(p.taxi_zone_id) if p.taxi_zone_id else r.taxi_zone_id,
            "taxi_zone_name": r.taxi_zone_name,
            "hour": hour,
            "busyness_level": p.busyness_level,
            "busyness_label": p.busyness_label,
            "busyness_score": p.busyness_score,
            "confidence": p.confidence,
            "taxi_dropoffs_1h": p.taxi_dropoffs_1h,
            "feature_source": p.feature_source,
        })

pred = pd.DataFrame(rows)

print("feature provenance:", pred.feature_source.value_counts().to_dict())
print("distinct scores per hour:",
      {h: int(pred[pred.hour == h].busyness_score.nunique()) for h in HOURS})
display(
    pred.groupby("hour").agg(
        venues=("busyness_score", "size"),
        mean=("busyness_score", "mean"),
        median=("busyness_score", "median"),
        confidence=("confidence", "mean"),
    ).round(4)
)
display(pd.crosstab(pred.hour, pred.busyness_label))

## 4. Basemap from the TLC taxi-zone shapefile

`taxi_zones.zip` ships with the inference container. Reading the Manhattan
polygons out of it gives a real island outline with no tile service, no API key
and no network call. Coordinates are NY State Plane (EPSG:2263), so they need
reprojecting to WGS84.

In [ ]:
CENTRAL_PARK_ZONE = 43

to_wgs = Transformer.from_crs("EPSG:2263", "EPSG:4326", always_xy=True)
sf = shapefile.Reader(str(FASTAPI_APP / "data" / "taxi_zones.zip"))
fields = [f[0] for f in sf.fields[1:]]
bi, zi = fields.index("borough"), fields.index("LocationID")

island, park = [], []
for shp, rec in zip(sf.shapes(), sf.records()):
    if rec[bi] != "Manhattan":
        continue
    bounds = list(shp.parts) + [len(shp.points)]
    for a, b in zip(bounds[:-1], bounds[1:]):
        ring = np.array(shp.points[a:b])
        if len(ring) < 3:
            continue
        lon, lat = to_wgs.transform(ring[:, 0], ring[:, 1])
        (park if int(rec[zi]) == CENTRAL_PARK_ZONE else island).append(
            np.column_stack([lon, lat])
        )

print(f"{len(island)} island rings, {len(park)} Central Park rings")

# Neighbourhood labels sit in the rivers, so they never fall under a marker.
# (text, lat, lon, horizontal alignment)
ANCHORS = [
    ("Inwood",             40.869, -73.948, "right"),
    ("Washington Hts.",    40.840, -73.966, "right"),
    ("Harlem",             40.812, -73.920, "left"),
    ("Upper West Side",    40.790, -74.002, "right"),
    ("Upper East Side",    40.772, -73.930, "left"),
    ("Midtown",            40.752, -73.947, "left"),
    ("Chelsea",            40.744, -74.020, "right"),
    ("Greenwich Village",  40.733, -73.967, "left"),
    ("Lower East Side",    40.714, -73.960, "left"),
    ("Financial District", 40.703, -73.985, "left"),
]

## 5. Draw the figure

One marker per restaurant, opaque, coloured by predicted class — deliberately
**not** a kernel-density blur, so a reader can count venues and see that the
evening shift happens across the whole island rather than in one hotspot.

In [ ]:
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.edgecolor": "#cccccc",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

fig, axes = plt.subplots(1, 3, figsize=(11.5, 8.6))

for ax, hour in zip(axes, HOURS):
    ax.set_facecolor("#eef3f7")                      # water

    for ring in island:
        ax.add_patch(MplPolygon(ring, closed=True, facecolor="#f4f2ee",
                                edgecolor="#dcd8d2", linewidth=0.4, zorder=1))
    for ring in park:
        ax.add_patch(MplPolygon(ring, closed=True, facecolor="#dde8d6",
                                edgecolor="#cbd9c2", linewidth=0.4, zorder=2))

    d = pred[pred.hour == hour]
    for level in (0, 1, 2):
        s = d[d.busyness_level == level]
        ax.scatter(s.longitude, s.latitude, s=26, c=CLASS_COLOURS[level],
                   edgecolors="white", linewidths=0.5, zorder=4)

    for text, la, lo, ha in ANCHORS:
        ax.text(lo, la, text, fontsize=6.5, color="#9a958e",
                ha=ha, va="center", style="italic", zorder=3)

    modal = d.busyness_level.mode()[0]
    share = 100 * (d.busyness_level == modal).mean()
    ax.set_title(
        f"{hour:02d}:00\n"
        f"mean {d.busyness_score.mean():.3f}  ·  {share:.0f}% {CLASS_NAMES[modal]}",
        fontsize=10.5, fontweight="bold", pad=10, linespacing=1.6,
    )

    ax.set_xlim(-74.028, -73.906)
    ax.set_ylim(40.694, 40.884)
    ax.set_aspect(1 / np.cos(np.deg2rad(40.78)))     # true shape at this latitude
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)

by_class = {
    lvl: [int((pred[pred.hour == h].busyness_level == lvl).sum()) for h in HOURS]
    for lvl in (0, 1, 2)
}
handles = [
    Line2D([], [], marker="o", linestyle="none", markersize=7,
           markerfacecolor=CLASS_COLOURS[lvl], markeredgecolor="white",
           label=f"{CLASS_NAMES[lvl]}  ({'/'.join(map(str, by_class[lvl]))})")
    for lvl in (0, 1, 2)
]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False,
           bbox_to_anchor=(0.5, 0.045), fontsize=9)
fig.text(0.5, 0.018,
         f"One marker per restaurant · {TARGET} venues, all "
         f"{picked.taxi_zone_id.nunique()} Manhattan taxi zones, minimum "
         f"{MIN_SEP:.0f} m apart · counts shown as 08:00/14:00/20:00",
         ha="center", fontsize=7.5, color="#666666")
fig.suptitle("Predicted restaurant busyness across Manhattan — Friday",
             fontsize=14, fontweight="bold", y=0.982)
fig.subplots_adjust(left=0.02, right=0.98, top=0.885, bottom=0.09, wspace=0.04)

fig.savefig(OUT_DIR / "fig_manhattan_busyness.png", dpi=300)
plt.show()
print("saved fig_manhattan_busyness.png to", OUT_DIR)

## 6. Numbers for the write-up

District rollup and the per-zone table, so the figures quoted in the paper come
from the same run that drew the panels.

In [ ]:
DISTRICTS = {
    "Lower Manhattan": [13, 87, 88, 209, 231, 261, 45, 232, 144, 211, 125],
    "Village & LES": [79, 4, 113, 114, 148, 249, 158],
    "Chelsea, Flatiron & Union Sq": [68, 90, 107, 234, 246, 224, 137],
    "Midtown": [48, 50, 100, 161, 162, 163, 164, 170, 186, 229, 230, 233],
    "Upper West Side": [24, 142, 143, 151, 238, 239, 166],
    "Upper East Side": [140, 141, 236, 237, 262, 263, 202],
    "Harlem": [41, 42, 74, 75, 152],
    "Upper Manhattan": [116, 127, 243, 244],
}
lookup = {z: name for name, zones in DISTRICTS.items() for z in zones}
pred["district"] = pred.taxi_zone_id.map(lookup)

print("mean predicted busyness score by district and hour")
display(pred.pivot_table(index="district", columns="hour",
                         values="busyness_score", aggfunc="mean").round(3))

print("share of venues predicted Severe Queue (%)")
display(pred.pivot_table(index="district", columns="hour",
                         values="busyness_level",
                         aggfunc=lambda s: 100 * (s == 2).mean()).round(1))

flip = pred.pivot_table(index="restaurant_id", columns="hour",
                        values="busyness_level")
n_flip = int(((flip[8] == 0) & (flip[20] == 2)).sum())
print(f"venues going No Wait at 08:00 -> Severe Queue at 20:00: {n_flip} of {TARGET}")

pred.to_csv(OUT_DIR / "manhattan_busyness_predictions.csv", index=False)
print("wrote manhattan_busyness_predictions.csv")

## Caveats

- The model's label is the **Google Places popular-times index**, a published
  activity proxy — not observed occupancy or queue length. The panels show
  predicted published activity.
- Held-out accuracy is 62.7% with QWK 0.599 on venues never seen in training.
  The model leans toward *overstating* waits (36.3% of its middle-class errors
  go up, 17.2% down), so read the red as an upper bound.
- Zone-level taxi demand alone explains almost none of the between-zone
  variation (*r* = 0.102, *R²* = 0.010). The spatial structure here comes from
  venue composition, not from the taxi feature.
- These 300 venues are **not** the contents of the seeded application database,
  which is confined to a ~350 m disc around Times Square. See the notebook
  header.